In [0]:
from pyspark.sql.functions import col, from_json, explode, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType

# 1. Base External Locations Paths
bronze_base_path = "abfss://bronze@staecommercegen2.dfs.core.windows.net/"
silver_base_path = "abfss://silver@staecommercegen2.dfs.core.windows.net/"

print("Starting Silver Layer Processing...\n")

# -------------------------------------------------------------------------
# A. PROCESS PRODUCTS DATA
# -------------------------------------------------------------------------
print("Processing Products Dataset...")
df_bronze_products = spark.read.format("delta").load(f"{bronze_base_path}raw_products")

df_silver_products = df_bronze_products.select(
    col("id").cast(IntegerType()).alias("product_id"),
    col("title").alias("product_name"),
    col("category"),
    col("brand"),
    col("price").cast(DoubleType()).alias("price"),
    col("discountPercentage").cast(DoubleType()).alias("discount_percentage"),
    col("rating").cast(DoubleType()).alias("rating"),
    col("stock").cast(IntegerType()).alias("stock_quantity"),
    col("ingestion_timestamp")
).filter(col("product_id").isNotNull())

df_silver_products.write.format("delta").mode("overwrite").save(f"{silver_base_path}dim_products")
print("✅ Saved dim_products to Silver Layer!\n")

# -------------------------------------------------------------------------
# B. PROCESS USERS DATA
# -------------------------------------------------------------------------
print("Processing Users Dataset...")
df_bronze_users = spark.read.format("delta").load(f"{bronze_base_path}raw_users")

df_silver_users = df_bronze_users.select(
    col("id").cast(IntegerType()).alias("user_id"),
    col("firstName").alias("first_name"),
    col("lastName").alias("last_name"),
    col("email"),
    col("phone"),
    col("gender"),
    col("age").cast(IntegerType()).alias("age"),
    col("ingestion_timestamp")
).filter(col("user_id").isNotNull())

df_silver_users.write.format("delta").mode("overwrite").save(f"{silver_base_path}dim_users")
print("✅ Saved dim_users to Silver Layer!\n")

# -------------------------------------------------------------------------
# C. PROCESS CARTS / ORDERS DATA (Flattening Nested Array)
# -------------------------------------------------------------------------
print("Processing Carts/Orders Dataset...")
df_bronze_carts = spark.read.format("delta").load(f"{bronze_base_path}raw_carts")

# Schema for parsing nested JSON string array in carts/products
cart_product_schema = ArrayType(
    StructType([
        StructField("id", IntegerType(), True),
        StructField("title", StringType(), True),
        StructField("price", DoubleType(), True),
        StructField("quantity", IntegerType(), True),
        StructField("total", DoubleType(), True),
        StructField("discountPercentage", DoubleType(), True),
        StructField("discountedTotal", DoubleType(), True)
    ])
)

# Parse string to JSON structure and explode products list
df_carts_parsed = df_bronze_carts.withColumn("products_arr", from_json(col("products"), cart_product_schema))
df_carts_exploded = df_carts_parsed.withColumn("product_item", explode(col("products_arr")))

df_silver_orders = df_carts_exploded.select(
    col("id").cast(IntegerType()).alias("cart_id"),
    col("userId").cast(IntegerType()).alias("user_id"),
    col("product_item.id").alias("product_id"),
    col("product_item.quantity").alias("quantity"),
    col("product_item.price").alias("unit_price"),
    col("product_item.total").alias("item_total_price"),
    col("product_item.discountedTotal").alias("item_discounted_price"),
    col("totalQuantity").cast(IntegerType()).alias("cart_total_quantity"),
    col("total").cast(DoubleType()).alias("cart_total_amount"),
    col("ingestion_timestamp")
)
#display(df_silver_orders)
df_silver_orders.write.format("delta").mode("overwrite").save(f"{silver_base_path}fact_orders")
print("✅ Saved fact_orders to Silver Layer!\n")

print("🚀 Silver Layer Transformation Complete!")

Starting Silver Layer Processing...

Processing Products Dataset...
✅ Saved dim_products to Silver Layer!

Processing Users Dataset...
✅ Saved dim_users to Silver Layer!

Processing Carts/Orders Dataset...
✅ Saved fact_orders to Silver Layer!

🚀 Silver Layer Transformation Complete!
